# Nettoyage & enrichissement des boxscores (minutes, DNP, totaux, ratios)

In [1]:
import pandas as pd
import numpy as np
import os
import datetime
from src.config import *
from src.utils import *
from src.feature_builder import *

In [2]:
#store start time of notebook
start_time = datetime.datetime.now()
print("Start time: ", start_time)

Start time:  2025-05-25 16:21:09.143782


# 🔁 Chargement des fichiers

In [3]:
boxscores_file = get_latest_file(DATA_LAST_BOXSCORES_BATCHES_MERGED_DIR)
games_file = get_latest_file(DATA_LAST_GAMES_MERGED_DIR)
df_boxscores = pd.read_csv(boxscores_file, dtype={'GAME_ID': str})
df_games = pd.read_csv(games_file, dtype={'GAME_ID': str})


# 🧼 Nettoyage des minutes jouées

In [4]:
def convert_minutes(val):
    if pd.isna(val) or val in ['DNP', '']:
        return 0.0
    try:
        parts = str(val).split(':')
        return int(parts[0]) + int(parts[1]) / 60 if len(parts) == 2 else float(val)
    except:
        return 0.0

df_boxscores['MINUTES_PLAYED'] = df_boxscores['MIN'].apply(convert_minutes)

# 🔄 Cast des colonnes numériques

In [5]:
cols_to_float = ['PTS', 'REB', 'AST', 'STL', 'BLK', 'TOV', 'FGM', 'FGA', 'FG3M', 'FG3A',
                 'FTM', 'FTA', 'OREB', 'DREB', 'PLUS_MINUS']
for col in cols_to_float:
    if col in df_boxscores.columns:
        df_boxscores[col] = pd.to_numeric(df_boxscores[col], errors='coerce').fillna(0)

# ⚙️ Aggrégation par équipe et match


In [6]:
# 3. Merge pour ajouter GAME_DATE à chaque ligne de boxscore
if 'GAME_DATE' not in df_boxscores.columns:
    # Pour éviter les doublons ou les merges en cascade si tu relances le code
    if 'GAME_DATE' in df_games.columns:
        df_boxscores = df_boxscores.merge(
            df_games[['GAME_ID', 'GAME_DATE']],
            on='GAME_ID',
            how='left'
        )
        # Conversion en datetime
        df_boxscores['GAME_DATE'] = pd.to_datetime(df_boxscores['GAME_DATE'])
    else:
        print("No GAME_DATE column found in games_df. Please check your merge operation.")


In [7]:

agg_cols = ['FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT',
            'OREB', 'DREB', 'REB', 'AST', 'STL', 'BLK', 'TO', 'PF', 'PTS', 'PLUS_MINUS', 'MINUTES_PLAYED']
team_match_stats = df_boxscores.groupby(['GAME_ID', 'TEAM_ID', 'GAME_DATE'])[agg_cols].sum().reset_index()
teams_in_game = df_boxscores.groupby('GAME_ID')['TEAM_ID'].unique().to_dict()
team_match_stats['OPP_TEAM_ID'] = team_match_stats.apply(
    lambda row: [tid for tid in teams_in_game[row['GAME_ID']] if tid != row['TEAM_ID']][0], axis=1)

# 🔁 Merge avec l’adversaire

In [8]:
team_cols = [col for col in team_match_stats.columns if col not in ['GAME_ID', 'TEAM_ID', 'OPP_TEAM_ID']]
opp_cols = [f"OPP_{col}" for col in team_cols]
df_opp = team_match_stats.rename(columns={col: f"OPP_{col}" for col in team_cols}).rename(
    columns={'TEAM_ID': 'OPP_TEAM_ID', 'OPP_TEAM_ID': 'TEAM_ID'})
match_dataset = pd.merge(
    team_match_stats,
    df_opp[['GAME_ID', 'TEAM_ID'] + opp_cols],
    on=['GAME_ID', 'TEAM_ID'],
    how='left'
)


In [9]:
match_dataset.columns

Index(['GAME_ID', 'TEAM_ID', 'GAME_DATE', 'FGM', 'FGA', 'FG_PCT', 'FG3M',
       'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB', 'REB', 'AST',
       'STL', 'BLK', 'TO', 'PF', 'PTS', 'PLUS_MINUS', 'MINUTES_PLAYED',
       'OPP_TEAM_ID', 'OPP_GAME_DATE', 'OPP_FGM', 'OPP_FGA', 'OPP_FG_PCT',
       'OPP_FG3M', 'OPP_FG3A', 'OPP_FG3_PCT', 'OPP_FTM', 'OPP_FTA',
       'OPP_FT_PCT', 'OPP_OREB', 'OPP_DREB', 'OPP_REB', 'OPP_AST', 'OPP_STL',
       'OPP_BLK', 'OPP_TO', 'OPP_PF', 'OPP_PTS', 'OPP_PLUS_MINUS',
       'OPP_MINUTES_PLAYED'],
      dtype='object')

In [10]:
match_dataset

,GAME_ID,TEAM_ID,GAME_DATE,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,...,OPP_DREB,OPP_REB,OPP_AST,OPP_STL,OPP_BLK,OPP_TO,OPP_PF,OPP_PTS,OPP_PLUS_MINUS,OPP_MINUTES_PLAYED
0,0020000001,1610612752,2000-10-31,50.0,140.0,8.906,6.0,22.0,2.666,38.0,...,58.0,74.0,54.0,20.0,10.0,26.0,48.0,202.0,290.0,480.000000
1,0020000001,1610612755,2000-10-31,76.0,132.0,10.382,6.0,16.0,3.000,44.0,...,46.0,74.0,28.0,12.0,8.0,44.0,60.0,144.0,-290.0,480.000000
2,0020000002,1610612739,2000-10-31,64.0,156.0,6.796,4.0,14.0,3.000,40.0,...,70.0,94.0,48.0,18.0,16.0,24.0,62.0,164.0,-40.0,480.000000
3,0020000002,1610612751,2000-10-31,62.0,170.0,6.524,6.0,20.0,2.334,34.0,...,82.0,104.0,32.0,10.0,16.0,38.0,54.0,172.0,40.0,480.000000
4,0020000003,1610612753,2000-10-31,68.0,158.0,7.900,12.0,32.0,2.750,46.0,...,70.0,88.0,40.0,12.0,2.0,52.0,56.0,172.0,-110.0,480.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64141,0052300131,1610612758,2024-04-16,86.0,196.0,8.150,36.0,78.0,5.796,28.0,...,68.0,84.0,38.0,10.0,6.0,32.0,34.0,188.0,-240.0,480.000000
64142,0052300201,1610612741,2024-04-19,70.0,184.0,8.816,26.0,86.0,4.130,16.0,...,76.0,94.0,52.0,12.0,12.0,30.0,22.0,224.0,210.0,480.000000
64143,0052300201,1610612748,2024-04-19,76.0,164.0,9.182,28.0,66.0,7.888,44.0,...,62.0,76.0,54.0,24.0,12.0,24.0,38.0,182.0,-210.0,480.000000
64144,0052300211,1610612740,2024-04-19,88.0,170.0,8.708,14.0,38.0,6.072,20.0,...,52.0,80.0,42.0,18.0,8.0,30.0,34.0,196.0,-70.0,480.000000


# 🏠 Ajout IS_HOME et IS_WIN


In [11]:


df_games['GAME_DATE'] = pd.to_datetime(df_games['GAME_DATE'])
match_dataset = match_dataset.merge(df_games[['GAME_ID', 'TEAM_ID', 'MATCHUP', 'SEASON']], on=['GAME_ID', 'TEAM_ID'], how='left')
match_dataset['IS_HOME'] = match_dataset['MATCHUP'].str.contains('vs').astype(int)
match_dataset['IS_WIN'] = (match_dataset['PTS'] > match_dataset['OPP_PTS']).astype(int)
match_dataset['POINT_DIFF'] = match_dataset['PTS'] - match_dataset['OPP_PTS']
match_dataset['GAME_DATE'] = pd.to_datetime(match_dataset['GAME_DATE'])
match_dataset = match_dataset.sort_values(['GAME_DATE', 'GAME_ID']).reset_index(drop=True)


In [12]:
match_dataset

,GAME_ID,TEAM_ID,GAME_DATE,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,...,OPP_TO,OPP_PF,OPP_PTS,OPP_PLUS_MINUS,OPP_MINUTES_PLAYED,MATCHUP,SEASON,IS_HOME,IS_WIN,POINT_DIFF
0,0020000001,1610612752,2000-10-31,50.0,140.0,8.906,6.0,22.0,2.666,38.0,...,26.0,48.0,202.0,290.0,480.0,NYK vs. PHI,2000-01,1,0,-58.0
1,0020000001,1610612755,2000-10-31,76.0,132.0,10.382,6.0,16.0,3.000,44.0,...,44.0,60.0,144.0,-290.0,480.0,PHI @ NYK,2000-01,0,1,58.0
2,0020000002,1610612739,2000-10-31,64.0,156.0,6.796,4.0,14.0,3.000,40.0,...,24.0,62.0,164.0,-40.0,480.0,CLE @ NJN,2000-01,0,1,8.0
3,0020000002,1610612751,2000-10-31,62.0,170.0,6.524,6.0,20.0,2.334,34.0,...,38.0,54.0,172.0,40.0,480.0,NJN vs. CLE,2000-01,1,0,-8.0
4,0020000003,1610612753,2000-10-31,68.0,158.0,7.900,12.0,32.0,2.750,46.0,...,52.0,56.0,172.0,-110.0,480.0,ORL vs. WAS,2000-01,1,1,22.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64141,0042400312,1610612760,2025-05-22,180.0,360.0,14.300,36.0,132.0,7.808,76.0,...,56.0,80.0,412.0,-300.0,0.0,OKC vs. MIN,2024-25,1,1,60.0
64142,0042400302,1610612752,2025-05-23,160.0,336.0,14.756,44.0,128.0,6.932,72.0,...,40.0,92.0,456.0,100.0,0.0,NYK vs. IND,2024-25,1,0,-20.0
64143,0042400302,1610612754,2025-05-23,172.0,332.0,17.144,52.0,120.0,13.936,60.0,...,48.0,76.0,436.0,-100.0,0.0,IND @ NYK,2024-25,0,1,20.0
64144,0042400313,1610612750,2025-05-24,220.0,384.0,32.280,80.0,160.0,25.432,52.0,...,56.0,80.0,404.0,-840.0,0.0,MIN vs. OKC,2024-25,1,1,168.0


# 🚀 Features avancées


In [13]:
N_LIST = [3, 5, 10, 25, 50, 100, 200]

match_dataset = compute_rolling_features(match_dataset, "TEAM_ID", ["TEAM_ID", "GAME_DATE"],
                                         ['PTS', 'REB', 'AST', 'FGM', 'FGA', 'FG_PCT', 'PLUS_MINUS'], N_LIST)
match_dataset = compute_winrates(match_dataset, "TEAM_ID", "IS_WIN", "IS_HOME", N_LIST)
match_dataset = compute_win_ratio(match_dataset, "TEAM_ID", "IS_WIN", N_LIST)

match_dataset["IS_WIN_SHIFTED"] = match_dataset.groupby("TEAM_ID")["IS_WIN"].shift(1).fillna(0).astype(int)
match_dataset["WIN_STREAK"] = match_dataset.groupby("TEAM_ID").apply(
    lambda x: compute_win_streak(x, "TEAM_ID", "IS_WIN_SHIFTED")).reset_index(level=0, drop=True)
match_dataset = compute_side_win_streak(match_dataset)

match_dataset["DAYS_SINCE_LAST_GAME"] = compute_rest_days(match_dataset, "GAME_DATE", "TEAM_ID")
match_dataset["OPP_DAYS_SINCE_LAST_GAME"] = compute_rest_days(match_dataset, "GAME_DATE", "OPP_TEAM_ID")
match_dataset["REST_ADVANTAGE"] = match_dataset["DAYS_SINCE_LAST_GAME"] - match_dataset["OPP_DAYS_SINCE_LAST_GAME"]

match_dataset = compute_rolling_rest_advantage(match_dataset, "TEAM_ID", "IS_HOME", "REST_ADVANTAGE", N_LIST)
match_dataset = compute_home_away_pts(match_dataset, "TEAM_ID", "IS_HOME", "PTS", "OPP_PTS", N_LIST)
match_dataset = rename_pts_against_columns(match_dataset)

match_dataset = compute_h2h(match_dataset, N_LIST)
match_dataset = compute_h2h_pts_margin(match_dataset, N_LIST)
match_dataset = compute_h2h_season(match_dataset)
match_dataset = compute_h2h_streak(match_dataset)

match_dataset = compute_elo(match_dataset)
match_dataset = compute_elo_season(match_dataset)


In [14]:
pd.options.display.max_columns = None
display(match_dataset)

,GAME_ID,TEAM_ID,GAME_DATE,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,STL,BLK,TO,PF,PTS,PLUS_MINUS,MINUTES_PLAYED,OPP_TEAM_ID,OPP_GAME_DATE,OPP_FGM,OPP_FGA,OPP_FG_PCT,OPP_FG3M,OPP_FG3A,OPP_FG3_PCT,OPP_FTM,OPP_FTA,OPP_FT_PCT,OPP_OREB,OPP_DREB,OPP_REB,OPP_AST,OPP_STL,OPP_BLK,OPP_TO,OPP_PF,OPP_PTS,OPP_PLUS_MINUS,OPP_MINUTES_PLAYED,MATCHUP,SEASON,IS_HOME,IS_WIN,POINT_DIFF,ROLL_PTS_3,ROLL_PTS_5,ROLL_PTS_10,ROLL_PTS_25,ROLL_PTS_50,ROLL_PTS_100,ROLL_PTS_200,ROLL_REB_3,ROLL_REB_5,ROLL_REB_10,ROLL_REB_25,ROLL_REB_50,ROLL_REB_100,ROLL_REB_200,ROLL_AST_3,ROLL_AST_5,ROLL_AST_10,ROLL_AST_25,ROLL_AST_50,ROLL_AST_100,ROLL_AST_200,ROLL_FGM_3,ROLL_FGM_5,ROLL_FGM_10,ROLL_FGM_25,ROLL_FGM_50,ROLL_FGM_100,ROLL_FGM_200,ROLL_FGA_3,ROLL_FGA_5,ROLL_FGA_10,ROLL_FGA_25,ROLL_FGA_50,ROLL_FGA_100,ROLL_FGA_200,ROLL_FG_PCT_3,ROLL_FG_PCT_5,ROLL_FG_PCT_10,ROLL_FG_PCT_25,ROLL_FG_PCT_50,ROLL_FG_PCT_100,ROLL_FG_PCT_200,ROLL_PLUS_MINUS_3,ROLL_PLUS_MINUS_5,ROLL_PLUS_MINUS_10,ROLL_PLUS_MINUS_25,ROLL_PLUS_MINUS_50,ROLL_PLUS_MINUS_100,ROLL_PLUS_MINUS_200,ROLL_HOME_WINRATE_3,ROLL_AWAY_WINRATE_3,ROLL_HOME_WINRATE_5,ROLL_AWAY_WINRATE_5,ROLL_HOME_WINRATE_10,ROLL_AWAY_WINRATE_10,ROLL_HOME_WINRATE_25,ROLL_AWAY_WINRATE_25,ROLL_HOME_WINRATE_50,ROLL_AWAY_WINRATE_50,ROLL_HOME_WINRATE_100,ROLL_AWAY_WINRATE_100,ROLL_HOME_WINRATE_200,ROLL_AWAY_WINRATE_200,ROLL_WIN_RATIO_3,ROLL_WIN_RATIO_5,ROLL_WIN_RATIO_10,ROLL_WIN_RATIO_25,ROLL_WIN_RATIO_50,ROLL_WIN_RATIO_100,ROLL_WIN_RATIO_200,WIN_STREAK,HOME_WIN_STREAK,AWAY_WIN_STREAK,DAYS_SINCE_LAST_GAME,OPP_DAYS_SINCE_LAST_GAME,REST_ADVANTAGE,ROLL_HOME_REST_ADV_3,ROLL_AWAY_REST_ADV_3,ROLL_HOME_REST_ADV_5,ROLL_AWAY_REST_ADV_5,ROLL_HOME_REST_ADV_10,ROLL_AWAY_REST_ADV_10,ROLL_HOME_REST_ADV_25,ROLL_AWAY_REST_ADV_25,ROLL_HOME_REST_ADV_50,ROLL_AWAY_REST_ADV_50,ROLL_HOME_REST_ADV_100,ROLL_AWAY_REST_ADV_100,ROLL_HOME_REST_ADV_200,ROLL_AWAY_REST_ADV_200,ROLL_HOME_PTS_FOR_3,ROLL_HOME_OPP_PTS_AGAINST_3,ROLL_AWAY_PTS_FOR_3,ROLL_AWAY_OPP_PTS_AGAINST_3,ROLL_HOME_PTS_FOR_5,ROLL_HOME_OPP_PTS_AGAINST_5,ROLL_AWAY_PTS_FOR_5,ROLL_AWAY_OPP_PTS_AGAINST_5,ROLL_HOME_PTS_FOR_10,ROLL_HOME_OPP_PTS_AGAINST_10,ROLL_AWAY_PTS_FOR_10,ROLL_AWAY_OPP_PTS_AGAINST_10,ROLL_HOME_PTS_FOR_25,ROLL_HOME_OPP_PTS_AGAINST_25,ROLL_AWAY_PTS_FOR_25,ROLL_AWAY_OPP_PTS_AGAINST_25,ROLL_HOME_PTS_FOR_50,ROLL_HOME_OPP_PTS_AGAINST_50,ROLL_AWAY_PTS_FOR_50,ROLL_AWAY_OPP_PTS_AGAINST_50,ROLL_HOME_PTS_FOR_100,ROLL_HOME_OPP_PTS_AGAINST_100,ROLL_AWAY_PTS_FOR_100,ROLL_AWAY_OPP_PTS_AGAINST_100,ROLL_HOME_PTS_FOR_200,ROLL_HOME_OPP_PTS_AGAINST_200,ROLL_AWAY_PTS_FOR_200,ROLL_AWAY_OPP_PTS_AGAINST_200,ROLL_HOME_PTS_AGAINST_3,ROLL_AWAY_PTS_AGAINST_3,ROLL_HOME_PTS_AGAINST_5,ROLL_AWAY_PTS_AGAINST_5,ROLL_HOME_PTS_AGAINST_10,ROLL_AWAY_PTS_AGAINST_10,ROLL_HOME_PTS_AGAINST_25,ROLL_AWAY_PTS_AGAINST_25,ROLL_HOME_PTS_AGAINST_50,ROLL_AWAY_PTS_AGAINST_50,ROLL_HOME_PTS_AGAINST_100,ROLL_AWAY_PTS_AGAINST_100,ROLL_HOME_PTS_AGAINST_200,ROLL_AWAY_PTS_AGAINST_200,H2H_LAST_3_DIFF,H2H_LAST_3_WINRATE,H2H_LAST_3_COUNT,H2H_LAST_5_DIFF,H2H_LAST_5_WINRATE,H2H_LAST_5_COUNT,H2H_LAST_10_DIFF,H2H_LAST_10_WINRATE,H2H_LAST_10_COUNT,H2H_LAST_25_DIFF,H2H_LAST_25_WINRATE,H2H_LAST_25_COUNT,H2H_LAST_50_DIFF,H2H_LAST_50_WINRATE,H2H_LAST_50_COUNT,H2H_LAST_100_DIFF,H2H_LAST_100_WINRATE,H2H_LAST_100_COUNT,H2H_LAST_200_DIFF,H2H_LAST_200_WINRATE,H2H_LAST_200_COUNT,H2H_LAST_3_PTS_FOR,H2H_LAST_3_PTS_AGAINST,H2H_LAST_3_MARGIN,H2H_LAST_5_PTS_FOR,H2H_LAST_5_PTS_AGAINST,H2H_LAST_5_MARGIN,H2H_LAST_10_PTS_FOR,H2H_LAST_10_PTS_AGAINST,H2H_LAST_10_MARGIN,H2H_LAST_25_PTS_FOR,H2H_LAST_25_PTS_AGAINST,H2H_LAST_25_MARGIN,H2H_LAST_50_PTS_FOR,H2H_LAST_50_PTS_AGAINST,H2H_LAST_50_MARGIN,H2H_LAST_100_PTS_FOR,H2H_LAST_100_PTS_AGAINST,H2H_LAST_100_MARGIN,H2H_LAST_200_PTS_FOR,H2H_LAST_200_PTS_AGAINST,H2H_LAST_200_MARGIN,H2H_SEASON_WINS,H2H_SEASON_MATCHES,H2H_SEASON_WINRATE,H2H_WIN_STREAK,ELO_PRE,OPP_ELO_PRE,ELO_PRE_SEASON,OPP_ELO_PRE_SEASON
0,0020000001,1610612752,2000-10-31,50.0,140.0,8.906,6.0,22.0,2.666,38.0,48.0,8.214,28.0,46.0,74.0,28.0,12.0,8.0,44.0,

# 🧽 Nettoyage et sauvegarde


In [15]:


final_date = datetime.datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
raw_save_path = os.path.join(DATA_FINAL_DATASET_DIR, f'nba_features_final_{final_date}.csv')
clean_save_path = os.path.join(DATA_FINAL_CLEANED_DATASET_DIR, f'nba_features_cleaned_final_{final_date}.csv')

match_dataset.to_csv(raw_save_path, index=False)


# Liste des colonnes à dropper (toutes les stats brutes et colonnes de match, identifiants inutiles, etc.)
drop_cols = [
    # Identifiants et logs
    "GAME_ID", "OPP_GAME_DATE", "MATCHUP", #"OPP_TEAM_ID", "GAME_DATE",

    "IS_WIN_SHIFTED", 

    # Stats brutes de match (pour les deux équipes)
    "FGM", "FGA", "FG_PCT", "FG3M", "FG3A", "FG3_PCT", "FTM", "FTA", "FT_PCT",
    "OREB", "DREB", "REB", "AST", "STL", "BLK", "TO", "PF", "PTS", "PLUS_MINUS", "MINUTES_PLAYED",

    "OPP_FGM", "OPP_FGA", "OPP_FG_PCT", "OPP_FG3M", "OPP_FG3A", "OPP_FG3_PCT", "OPP_FTM", "OPP_FTA", "OPP_FT_PCT",
    "OPP_OREB", "OPP_DREB", "OPP_REB", "OPP_AST", "OPP_STL", "OPP_BLK", "OPP_TO", "OPP_PF", "OPP_PTS",
    "OPP_PLUS_MINUS", "OPP_MINUTES_PLAYED",
    "POINT_DIFF"
]

# Droppage effectif
final_dataset = match_dataset.drop(columns=[col for col in drop_cols if col in match_dataset.columns])


final_cleaned = match_dataset.drop(columns=drop_cols, errors='ignore')
final_cleaned.to_csv(clean_save_path, index=False)

print(f"✅ Fichier brut : {raw_save_path}")
print(f"✅ Fichier clean : {clean_save_path}")

✅ Fichier brut : data/final_dataset/nba_features_final_2025-05-25_16-22-56.csv
✅ Fichier clean : data/final_cleaned_dataset/nba_features_cleaned_final_2025-05-25_16-22-56.csv


In [16]:
final_cleaned

,TEAM_ID,GAME_DATE,OPP_TEAM_ID,SEASON,IS_HOME,IS_WIN,ROLL_PTS_3,ROLL_PTS_5,ROLL_PTS_10,ROLL_PTS_25,ROLL_PTS_50,ROLL_PTS_100,ROLL_PTS_200,ROLL_REB_3,ROLL_REB_5,ROLL_REB_10,ROLL_REB_25,ROLL_REB_50,ROLL_REB_100,ROLL_REB_200,ROLL_AST_3,ROLL_AST_5,ROLL_AST_10,ROLL_AST_25,ROLL_AST_50,ROLL_AST_100,ROLL_AST_200,ROLL_FGM_3,ROLL_FGM_5,ROLL_FGM_10,ROLL_FGM_25,ROLL_FGM_50,ROLL_FGM_100,ROLL_FGM_200,ROLL_FGA_3,ROLL_FGA_5,ROLL_FGA_10,ROLL_FGA_25,ROLL_FGA_50,ROLL_FGA_100,ROLL_FGA_200,ROLL_FG_PCT_3,ROLL_FG_PCT_5,ROLL_FG_PCT_10,ROLL_FG_PCT_25,ROLL_FG_PCT_50,ROLL_FG_PCT_100,ROLL_FG_PCT_200,ROLL_PLUS_MINUS_3,ROLL_PLUS_MINUS_5,ROLL_PLUS_MINUS_10,ROLL_PLUS_MINUS_25,ROLL_PLUS_MINUS_50,ROLL_PLUS_MINUS_100,ROLL_PLUS_MINUS_200,ROLL_HOME_WINRATE_3,ROLL_AWAY_WINRATE_3,ROLL_HOME_WINRATE_5,ROLL_AWAY_WINRATE_5,ROLL_HOME_WINRATE_10,ROLL_AWAY_WINRATE_10,ROLL_HOME_WINRATE_25,ROLL_AWAY_WINRATE_25,ROLL_HOME_WINRATE_50,ROLL_AWAY_WINRATE_50,ROLL_HOME_WINRATE_100,ROLL_AWAY_WINRATE_100,ROLL_HOME_WINRATE_200,ROLL_AWAY_WINRATE_200,ROLL_WIN_RATIO_3,ROLL_WIN_RATIO_5,ROLL_WIN_RATIO_10,ROLL_WIN_RATIO_25,ROLL_WIN_RATIO_50,ROLL_WIN_RATIO_100,ROLL_WIN_RATIO_200,WIN_STREAK,HOME_WIN_STREAK,AWAY_WIN_STREAK,DAYS_SINCE_LAST_GAME,OPP_DAYS_SINCE_LAST_GAME,REST_ADVANTAGE,ROLL_HOME_REST_ADV_3,ROLL_AWAY_REST_ADV_3,ROLL_HOME_REST_ADV_5,ROLL_AWAY_REST_ADV_5,ROLL_HOME_REST_ADV_10,ROLL_AWAY_REST_ADV_10,ROLL_HOME_REST_ADV_25,ROLL_AWAY_REST_ADV_25,ROLL_HOME_REST_ADV_50,ROLL_AWAY_REST_ADV_50,ROLL_HOME_REST_ADV_100,ROLL_AWAY_REST_ADV_100,ROLL_HOME_REST_ADV_200,ROLL_AWAY_REST_ADV_200,ROLL_HOME_PTS_FOR_3,ROLL_HOME_OPP_PTS_AGAINST_3,ROLL_AWAY_PTS_FOR_3,ROLL_AWAY_OPP_PTS_AGAINST_3,ROLL_HOME_PTS_FOR_5,ROLL_HOME_OPP_PTS_AGAINST_5,ROLL_AWAY_PTS_FOR_5,ROLL_AWAY_OPP_PTS_AGAINST_5,ROLL_HOME_PTS_FOR_10,ROLL_HOME_OPP_PTS_AGAINST_10,ROLL_AWAY_PTS_FOR_10,ROLL_AWAY_OPP_PTS_AGAINST_10,ROLL_HOME_PTS_FOR_25,ROLL_HOME_OPP_PTS_AGAINST_25,ROLL_AWAY_PTS_FOR_25,ROLL_AWAY_OPP_PTS_AGAINST_25,ROLL_HOME_PTS_FOR_50,ROLL_HOME_OPP_PTS_AGAINST_50,ROLL_AWAY_PTS_FOR_50,ROLL_AWAY_OPP_PTS_AGAINST_50,ROLL_HOME_PTS_FOR_100,ROLL_HOME_OPP_PTS_AGAINST_100,ROLL_AWAY_PTS_FOR_100,ROLL_AWAY_OPP_PTS_AGAINST_100,ROLL_HOME_PTS_FOR_200,ROLL_HOME_OPP_PTS_AGAINST_200,ROLL_AWAY_PTS_FOR_200,ROLL_AWAY_OPP_PTS_AGAINST_200,ROLL_HOME_PTS_AGAINST_3,ROLL_AWAY_PTS_AGAINST_3,ROLL_HOME_PTS_AGAINST_5,ROLL_AWAY_PTS_AGAINST_5,ROLL_HOME_PTS_AGAINST_10,ROLL_AWAY_PTS_AGAINST_10,ROLL_HOME_PTS_AGAINST_25,ROLL_AWAY_PTS_AGAINST_25,ROLL_HOME_PTS_AGAINST_50,ROLL_AWAY_PTS_AGAINST_50,ROLL_HOME_PTS_AGAINST_100,ROLL_AWAY_PTS_AGAINST_100,ROLL_HOME_PTS_AGAINST_200,ROLL_AWAY_PTS_AGAINST_200,H2H_LAST_3_DIFF,H2H_LAST_3_WINRATE,H2H_LAST_3_COUNT,H2H_LAST_5_DIFF,H2H_LAST_5_WINRATE,H2H_LAST_5_COUNT,H2H_LAST_10_DIFF,H2H_LAST_10_WINRATE,H2H_LAST_10_COUNT,H2H_LAST_25_DIFF,H2H_LAST_25_WINRATE,H2H_LAST_25_COUNT,H2H_LAST_50_DIFF,H2H_LAST_50_WINRATE,H2H_LAST_50_COUNT,H2H_LAST_100_DIFF,H2H_LAST_100_WINRATE,H2H_LAST_100_COUNT,H2H_LAST_200_DIFF,H2H_LAST_200_WINRATE,H2H_LAST_200_COUNT,H2H_LAST_3_PTS_FOR,H2H_LAST_3_PTS_AGAINST,H2H_LAST_3_MARGIN,H2H_LAST_5_PTS_FOR,H2H_LAST_5_PTS_AGAINST,H2H_LAST_5_MARGIN,H2H_LAST_10_PTS_FOR,H2H_LAST_10_PTS_AGAINST,H2H_LAST_10_MARGIN,H2H_LAST_25_PTS_FOR,H2H_LAST_25_PTS_AGAINST,H2H_LAST_25_MARGIN,H2H_LAST_50_PTS_FOR,H2H_LAST_50_PTS_AGAINST,H2H_LAST_50_MARGIN,H2H_LAST_100_PTS_FOR,H2H_LAST_100_PTS_AGAINST,H2H_LAST_100_MARGIN,H2H_LAST_200_PTS_FOR,H2H_LAST_200_PTS_AGAINST,H2H_LAST_200_MARGIN,H2H_SEASON_WINS,H2H_SEASON_MATCHES,H2H_SEASON_WINRATE,H2H_WIN_STREAK,ELO_PRE,OPP_ELO_PRE,ELO_PRE_SEASON,OPP_ELO_PRE_SEASON
0,1610612752,2000-10-31,1610612755,2000-01,1,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,7.0,-8880.0,8887.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Na

In [17]:
#display full columns
pd.set_option('display.max_column', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_seq_items', None)
pd.set_option('display.max_colwidth', 500)
pd.set_option('expand_frame_repr', True)
# Display the final cleaned dataset
print("Final cleaned dataset:")
toto = final_cleaned.columns
print(toto)


Final cleaned dataset:
Index(['TEAM_ID', 'GAME_DATE', 'OPP_TEAM_ID', 'SEASON', 'IS_HOME', 'IS_WIN',
       'ROLL_PTS_3', 'ROLL_PTS_5', 'ROLL_PTS_10', 'ROLL_PTS_25', 'ROLL_PTS_50',
       'ROLL_PTS_100', 'ROLL_PTS_200', 'ROLL_REB_3', 'ROLL_REB_5',
       'ROLL_REB_10', 'ROLL_REB_25', 'ROLL_REB_50', 'ROLL_REB_100',
       'ROLL_REB_200', 'ROLL_AST_3', 'ROLL_AST_5', 'ROLL_AST_10',
       'ROLL_AST_25', 'ROLL_AST_50', 'ROLL_AST_100', 'ROLL_AST_200',
       'ROLL_FGM_3', 'ROLL_FGM_5', 'ROLL_FGM_10', 'ROLL_FGM_25', 'ROLL_FGM_50',
       'ROLL_FGM_100', 'ROLL_FGM_200', 'ROLL_FGA_3', 'ROLL_FGA_5',
       'ROLL_FGA_10', 'ROLL_FGA_25', 'ROLL_FGA_50', 'ROLL_FGA_100',
       'ROLL_FGA_200', 'ROLL_FG_PCT_3', 'ROLL_FG_PCT_5', 'ROLL_FG_PCT_10',
       'ROLL_FG_PCT_25', 'ROLL_FG_PCT_50', 'ROLL_FG_PCT_100',
       'ROLL_FG_PCT_200', 'ROLL_PLUS_MINUS_3', 'ROLL_PLUS_MINUS_5',
       'ROLL_PLUS_MINUS_10', 'ROLL_PLUS_MINUS_25', 'ROLL_PLUS_MINUS_50',
       'ROLL_PLUS_MINUS_100', 'ROLL_PLUS_MINUS_200', 'ROL

In [18]:
#store end time of notebook
end_time = datetime.datetime.now()
print("End time: ", end_time)
print("Total time: ", end_time - start_time)

End time:  2025-05-25 16:23:21.368346
Total time:  0:02:12.224564
